In [3]:
# ============================================
# TAHAP 4: INFERENCE (PREDIKSI SENTIMEN) - PERBAIKAN
# ============================================

# ========== LANGKAH 1: INSTALL LIBRARY YANG DIPERLUKAN ==========
!pip install tensorflow pandas numpy scikit-learn nltk sastrawi -q

# ========== LANGKAH 2: LOAD MODEL DAN DEPENDENSI ==========
import tensorflow as tf
import pickle
import re
import string
import numpy as np
import nltk
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Download stopword NLTK
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords

# Import Sastrawi
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

print("📂 Memuat model dan tokenizer...")

# Load model
model = tf.keras.models.load_model('sentiment_model_best.h5')
print("✅ Model berhasil dimuat")

# Load tokenizer
with open('tokenizer.pkl', 'rb') as f:
    tokenizer = pickle.load(f)
print("✅ Tokenizer berhasil dimuat")

# Load label encoder
with open('label_encoder.pkl', 'rb') as f:
    label_encoder = pickle.load(f)
print("✅ Label encoder berhasil dimuat")

# Parameter (harus SAMA dengan training)
MAX_SEQUENCE_LEN = 100

# ========== LANGKAH 3: FUNGSI PREPROCESSING (SAMA SEPERTI TRAINING) ==========

def clean_text(text):
    """
    Membersihkan teks input
    """
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'http\S+|www\S+|https\S+|bit\.ly\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Stopword (sama dengan training)
indonesian_stopwords = set(stopwords.words('indonesian'))
custom_stopwords = {'yg', 'dgn', 'jd', 'klo', 'sdh', 'udah', 'gak', 'ga', 'nggak',
                    'nya', 'si', 'nih', 'dong', 'sih', 'deh', 'yah', 'ah', 'kok',
                    'kan', 'lah', 'pun', 'dan', 'juga', 'saya', 'kamu', 'dia', 'mereka',
                    'kau', 'aku', 'gua', 'gue', 'lo', 'lu', 'anda', 'kalian'}
indonesian_stopwords.update(custom_stopwords)

def remove_stopwords(text):
    words = text.split()
    words = [w for w in words if w not in indonesian_stopwords and len(w) > 1]
    return ' '.join(words)

# Stemming
factory = StemmerFactory()
stemmer = factory.create_stemmer()

def stem_text(text):
    if len(text) < 1000:
        return stemmer.stem(text)
    return text

def preprocess_input(text):
    """
    Pipeline preprocessing untuk input baru
    """
    text = clean_text(text)
    text = remove_stopwords(text)
    text = stem_text(text)
    return text

# ========== LANGKAH 4: FUNGSI PREDIKSI SENTIMEN ==========
def predict_sentiment(text):
    """
    Memprediksi sentimen dari teks input
    Output: (sentimen_string, confidence)
    """
    # 1. Preprocessing
    cleaned = preprocess_input(text)

    # 2. Konversi ke sequence menggunakan tokenizer yang sama
    sequence = tokenizer.texts_to_sequences([cleaned])

    # 3. Padding
    padded = pad_sequences(sequence, maxlen=MAX_SEQUENCE_LEN, padding='post', truncating='post')

    # 4. Prediksi dengan model
    prediction = model.predict(padded, verbose=0)

    # 5. Ambil kelas dengan probabilitas tertinggi
    predicted_class_index = np.argmax(prediction[0])

    # 6. Konversi ke label asli
    sentiment = label_encoder.inverse_transform([predicted_class_index])[0]

    # 7. Dapatkan confidence/probabilitas
    confidence = float(prediction[0][predicted_class_index])

    return sentiment, confidence

# ========== LANGKAH 5: UJI COBA DENGAN BERBAGAI INPUT ==========
print("\n" + "="*60)
print("🔮 UJI COBA PREDIKSI SENTIMEN")
print("="*60)

# Daftar teks uji (negatif, netral, positif)
test_cases = [
    # Negatif
    "Printer ini jelek banget, gampang rusak dan tinta boros",
    "Kecewa berat sama produk ini, tidak sesuai ekspektasi",
    "Layanan pelanggannya buruk, tidak pernah merespon",

    # Netral
    "Harganya standar saja, tidak murah tidak mahal",
    "Printer ini bisa dipakai untuk mencetak dokumen sehari-hari",
    "Pengiriman 3 hari sampai, biasa saja",

    # Positif
    "Produknya bagus banget! Print nya jernih dan cepat",
    "Rekomendasi printer terbaik untuk rumahan, worth it!",
    "Mantap! Gak nyesel beli ini, hasil cetakannya bagus"
]

print("\n📊 HASIL PREDIKSI:")
print("-" * 70)
print(f"{'No':<3} {'Teks':<45} {'Sentimen':<12} {'Confidence':<10}")
print("-" * 70)

for i, text in enumerate(test_cases, 1):
    sentiment, confidence = predict_sentiment(text)
    # Potong teks jika terlalu panjang
    display_text = text[:42] + "..." if len(text) > 45 else text
    print(f"{i:<3} {display_text:<45} {sentiment:<12} {confidence*100:.1f}%")

print("-" * 70)

# ========== LANGKAH 6: INPUT INTERAKTIF (UNTUK DEMO) ==========
print("\n" + "="*60)
print("💬 MODE INTERAKTIF")
print("="*60)
print("Ketik komentar Anda (atau 'quit' untuk keluar):")

while True:
    user_input = input("\n📝 Masukkan komentar: ")
    if user_input.lower() == 'quit':
        print("Terima kasih! Selesai.")
        break

    if user_input.strip() == "":
        print("⚠️ Masukkan teks yang valid.")
        continue

    sentiment, confidence = predict_sentiment(user_input)

    # Tampilkan dengan emoji
    if sentiment == 'positif':
        emoji = "😊"
        color = "\033[92m"  # Hijau (terminal)
    elif sentiment == 'negatif':
        emoji = "😠"
        color = "\033[91m"  # Merah
    else:
        emoji = "😐"
        color = "\033[93m"  # Kuning

    print(f"{emoji} Sentimen: {color}{sentiment.upper()}\033[0m (confidence: {confidence*100:.1f}%)")

# ========== LANGKAH 7: BUKTI INFERENSI ==========
print("\n" + "="*60)
print("📸 BUKTI INFERENSI - SIMPAN SEBAGAI SCREENSHOT")
print("="*60)

# Buat batch prediksi untuk dokumentasi
batch_texts = [
    "Saya suka banget sama produk ini, rekomendasi!",
    "Pengiriman lama banget, mengecewakan",
    "Produk biasa saja, tidak ada yang spesial"
]

print("\n📋 Contoh prediksi batch (screenshot area di bawah ini):")
print("-" * 60)
for text in batch_texts:
    sent, conf = predict_sentiment(text)
    print(f"\n📝 Teks: {text}")
    print(f"🎯 Hasil: {sent.upper()} (confidence: {conf*100:.1f}%)")
print("\n" + "-" * 60)
print("✅ Ambil screenshot dari output di atas sebagai bukti inference.")

# ========== LANGKAH 8: SIMPAN HASIL INFERENSI ==========
import pandas as pd

results_list = []
for text in test_cases:
    sentiment, confidence = predict_sentiment(text)
    results_list.append({
        'text': text,
        'sentiment': sentiment,
        'confidence': confidence
    })

df_results = pd.DataFrame(results_list)
df_results.to_csv('inference_results.csv', index=False)
print("\n✅ Hasil inference disimpan ke 'inference_results.csv'")

# Tampilkan ringkasan
print("\n" + "="*60)
print("📊 RINGKASAN INFERENSI")
print("="*60)
print(df_results.to_string(index=False))

print("\n" + "="*60)
print("🎉 INFERENCE SELESAI! MODEL SIAP DI-SUBMIT")
print("="*60)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.7/209.7 kB 4.9 MB/s eta 0:00:00


📂 Memuat model dan tokenizer...
✅ Model berhasil dimuat
✅ Tokenizer berhasil dimuat
✅ Label encoder berhasil dimuat

🔮 UJI COBA PREDIKSI SENTIMEN

📊 HASIL PREDIKSI:
----------------------------------------------------------------------
No  Teks                                          Sentimen     Confidence
----------------------------------------------------------------------
1   Printer ini jelek banget, gampang rusak da... negatif      100.0%
2   Kecewa berat sama produk ini, tidak sesuai... negatif      100.0%
3   Layanan pelanggannya buruk, tidak pernah m... negatif      92.6%
4   Harganya standar saja, tidak murah tidak m... positif      59.9%
5   Printer ini bisa dipakai untuk mencetak do... netral       100.0%
6   Pengiriman 3 hari sampai, biasa saja          netral       100.0%
7   Produknya bagus banget! Print nya jernih d... positif      100.0%
8   Rekomendasi printer terbaik untuk rumahan,... positif      100.0%
9   Mantap! Gak nyesel beli ini, hasil cetakan... positif    